# S3 drill — OUT-OF-SAMPLE: June 2025 and June 2026 recons (OPENED ONCE)

Applies the frozen primary rule (P2 in `research/screens/S3_DRILL_PREREG.md`) to two events and nothing else.
No grid and no re-tuning. The notebook refuses to recompute once `OOS_RESULT.json` exists.

- **Data opened:** IJR and IWM adjusted closes only, 2025-01-01 .. 2026-09-30, pulled to `reference/oos/`. The
  in-sample notebooks never read that folder. The stock universe, SPY and RSP stay sealed for 2025+.
- **Rule:** entry at the recon close, exit at recon + 20 bars. Spread = compounded IJR − IWM. Baseline = mean of all
  other non-overlapping 20-bar windows starting in the same year (2026 is a partial year, through 2026-09-30).
- **Per event:** raw spread, excess, excess net of base cost (5bp round trip + 0.5%/yr borrow × 20/252), percentile.
- **OOS read (pre-registered):** *supports* = both events have excess net of base cost > 0; *mixed* = one does;
  *against* = neither. Raw spread > 0 is reported alongside.

In [ ]:
# ===== Cell 0 — config (FROZEN — must equal P2) =====
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass                                            # local smoke test: BUKOWSKI_DIR points at fixtures
import pandas as pd, numpy as np, glob, json, os, math, urllib.request
from pathlib import Path
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

DATA_DIR = Path(os.environ.get('BUKOWSKI_DIR', '/content/drive/MyDrive/Bukowski'))
REF_DIR  = DATA_DIR/'reference'
OUT_DIR  = DATA_DIR/'results'/'screens'; OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- LOCKBOX ----
CUT_DATE = pd.Timestamp('2024-12-31')     # every series cut at load; nothing after this is read, pulled or printed

OOS_START, OOS_END = pd.Timestamp('2025-01-01'), pd.Timestamp('2026-09-30')
FROZEN = {'entry': 0, 'horizon': 20, 'rt_cost': 5e-4, 'borrow': 0.005}
RECON_OOS = {2025: '2025-06-27', 2026: '2026-06-26'}
OOS_REF = REF_DIR/'oos'
O_OUT = OUT_DIR/'s3_drill_oos'; O_OUT.mkdir(parents=True, exist_ok=True)
MARKER = O_OUT/'OOS_RESULT.json'

In [ ]:
# ===== Cell 1 — guards: opened once, in-sample null check passed =====
if MARKER.exists():
    print(open(MARKER).read())
    raise RuntimeError('OOS already opened once — result above. Do not recompute.')
nc = json.load(open(OUT_DIR/'s3_drill'/'null_check.json'))
assert nc['null_ok'], 'in-sample null check did not pass — OOS stays sealed'
dv = json.load(open(OUT_DIR/'s3_drill'/'drill_verdict.json'))
print('in-sample drill verdict:', 'PASS' if dv['drill_pass'] else 'FAIL', dv['criteria'])


def rule_recon(y):
    """June recon: last Friday of June, one week earlier if that Friday is the 29th/30th."""
    fr = [d for d in pd.date_range(f'{y}-06-01', f'{y}-06-30') if d.weekday() == 4][-1]
    return fr - pd.Timedelta(days=7) if fr.day >= 29 else fr


for y, d in RECON_OOS.items():
    assert pd.Timestamp(d) == rule_recon(y), (y, d, rule_recon(y))

In [ ]:
# ===== Cell 2 — pull IJR/IWM 2025-01-01 .. 2026-09-30 into reference/oos/ =====
def pull_oos(tk):
    path = OOS_REF/f'{tk}_{OOS_START:%Y%m%d}_{OOS_END:%Y%m%d}.csv'
    if path.exists():
        print(f'{tk}: present at {path}'); return path
    token = os.environ.get('TIINGO_API_KEY')
    if not token:
        from google.colab import userdata; token = userdata.get('TIINGO_API_KEY')
    url = (f'https://api.tiingo.com/tiingo/daily/{tk}/prices?startDate={OOS_START:%Y-%m-%d}'
           f'&endDate={OOS_END:%Y-%m-%d}&token={token}')
    js = pd.DataFrame(json.load(urllib.request.urlopen(url, timeout=60)))
    df = pd.DataFrame({'Date': pd.to_datetime(js['date']).dt.strftime('%Y-%m-%d'), 'Close': js['adjClose']})
    OOS_REF.mkdir(parents=True, exist_ok=True); df.to_csv(path, index=False)
    print(f'{tk}: pulled {len(df)} rows -> {path}'); return path

OPX = {}
for tk in ['IJR', 'IWM']:
    d = pd.read_csv(pull_oos(tk)); d['Date'] = pd.to_datetime(d['Date'])
    OPX[tk] = d.drop_duplicates('Date').set_index('Date')['Close'].sort_index()

In [ ]:
# ===== Cell 3 — the one OOS computation =====
assert not MARKER.exists()
px = pd.concat(OPX, axis=1).dropna(); px = px[(px.index >= OOS_START) & (px.index <= OOS_END)]
D = px.index; A, B = px['IJR'].to_numpy(float), px['IWM'].to_numpy(float); YEAR = D.year.to_numpy()
e, H = FROZEN['entry'], FROZEN['horizon']; L = H - e
c = FROZEN['rt_cost'] + FROZEN['borrow'] * L / 252
sp = A[L:] / A[:-L] - B[L:] / B[:-L]; yr = YEAR[:len(sp)]
rows = []
for y, d in RECON_OOS.items():
    i0 = D.get_loc(pd.Timestamp(d)); s, t = i0 + e, i0 + H
    assert t < len(D), f'{y}: exit bar beyond {OOS_END.date()}'
    idx = np.where(yr == y)[0]; idx = idx[(idx + L < s) | (idx > t)]
    ev, base = sp[s], sp[idx].mean()
    rows.append({'year': y, 'recon': d, 'exit': str(D[t].date()), 'spread': ev, 'base': base, 'excess': ev - base,
                 'excess_net_base': ev - base - c, 'pct': 100 * (sp[idx] < ev).mean(), 'n_base': len(idx),
                 'raw_pos': bool(ev > 0), 'net_excess_pos': bool(ev - base - c > 0)})
OOS = pd.DataFrame(rows).set_index('year')
k = int(OOS['net_excess_pos'].sum())
READ = {2: 'supports', 1: 'mixed', 0: 'against'}[k]
print(OOS.to_string()); print(f'\nOOS read (pre-registered): {READ}  ({k}/2 events with excess net of base cost > 0)')
OOS.to_csv(O_OUT/'oos_events.csv')
today = pd.Timestamp.now(tz='UTC')
json.dump({'opened_utc': today.isoformat(), 'frozen': FROZEN, 'read': READ,
           'events': json.loads(OOS.reset_index().to_json(orient='records'))}, open(MARKER, 'w'), indent=1)
print('\nPaste into P3 of S3_DRILL_PREREG.md:')
for y, r in OOS.iterrows():
    print(f'| June {y} recon (OOS, frozen rule) | {today.date()} | {r.spread:+.2%} | '
          f'{"yes" if r.raw_pos else "no"} (raw > 0) | excess {r.excess:+.2%}, net {r.excess_net_base:+.2%}, '
          f'pct {r.pct:.0f}; OOS read: {READ} | |')